# Depth-dependent Gradient × Update routers

**One experiment, four steps:**

1. Train a shared scorer and two task corrections.
2. Compare shared, specialized, and depth-dependent rankings.
3. Apply actual FFN skipping — **implemented in this notebook**.
4. Measure output quality and inference time — **not implemented yet**.

This notebook implements **1–3**. All notebook helpers are included below; **no `depth_router/` folder is needed**. Use your existing Show-o2 Colab setup, model checkpoints, and light generation collector.

## 1. Train the scorers
### 1.1 Open the existing environment

Select a GPU runtime and run the original Show-o2 setup first. Generation features are reconstructed from the saved light files using forward passes; gradients are not collected again.

In [ ]:
import os
import sys
import gc
import json
import hashlib
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    REPO_ROOT = Path.cwd()
else:
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/FlashU-Lab")

if not (REPO_ROOT / "evaluation/calculate_gradient_update_generation_light.py").exists():
    raise FileNotFoundError("Set REPO_ROOT to your existing FlashU-Lab repository with the light collector.")
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

### 1.2 Check paths and settings

- Training uses **`log(raw Gradient × Update, clamped at 1e-12)`**, matching the previous training objective. Evaluation ranks the raw scores.
- Keep **32 random valid tokens per example/layer/step** in RAM for training and validation loss. This is sampling, not lossless compression. Evaluation uses **all valid tokens**.
- Boundary **14** is a fixed pilot choice. Layers **0–26** are scored; layer **27** will stay dense.
- G targets use the dense model's generated reference image, not an external ground-truth image. Keep the original model checkpoints available for replay.

In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/intership")
U_ROOT = DATA_ROOT / "gradient_update_understanding"
G_ROOT = DATA_ROOT / "gradient_update_generation/counting_spatial"
OUTPUT_DIR = DATA_ROOT / "gradient_mlp_results/depth_router_minimal_v1"

U_CATEGORIES = ["counting", "spatial_relations", "attribute_combinations", "visual_state_traffic_lights"]
SEEDS = [42, 43]
SPLIT_SEED, TOKEN_SEED = 42, 123
TOKENS_PER_GROUP = 32
LAYERS = tuple(range(27))
DEPTH_START = 14
TOP_FRACTION = 0.20
KEEP_FRACTION = 0.80  # Future FFN routing; not the overlap metric.
TRAIN = dict(hidden_width=128, batch_size=256, max_epochs=15, updates=150, patience=3, lr=1e-3)
WEIGHTS = OUTPUT_DIR / "routers.pt"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 Load data and preserve the split

The following cell contains the data functions. It reads saved U features and reconstructs G features one step at a time.

Original assignments stay fixed: **U = 16 train / 4 validation / 20 test; G = 8 / 2 / 30**. The added 16 U and 28 G examples are test-only. Every example's tokens and steps stay together.

In [ ]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import torch


def load_saved(path):
    return torch.load(path, map_location="cpu", weights_only=True, mmap=True)


def discover_data(U_ROOT, G_ROOT, U_CATEGORIES, SPLIT_SEED=42):
    expected_order = ["counting", "spatial_relations", "attribute_combinations",
                      "visual_state_traffic_lights"]
    if list(U_CATEGORIES) != expected_order:
        raise ValueError("Keep the category order to preserve the original split.")
    folders = [("U", name, Path(U_ROOT) / name, 10) for name in U_CATEGORIES]
    folders.append(("G", "counting_spatial", Path(G_ROOT), 40))
    examples, rng = [], np.random.default_rng(SPLIT_SEED)
    for task, category, folder, expected_count in folders:
        run_file = folder / "run.json"
        run = json.loads(run_file.read_text(encoding="utf-8"))
        expected_score = ("token_gradient_update_answer_loss_v1" if task == "U"
                          else "token_gradient_update_flow_loss_v1")
        if run.get("status") != "complete" or run.get("score_name") != expected_score:
            raise ValueError(f"{run_file}: incomplete collection or wrong score type.")
        paths = sorted(folder.glob("sample_*.pt"))
        if len(paths) != expected_count:
            raise ValueError(f"{folder}: expected {expected_count} samples, found {len(paths)}.")
        group = []
        for index, path in enumerate(paths):
            payload = load_saved(path)
            sample, layers = payload["sample"], payload["layer_indices"]
            if (payload.get("schema_version") != 1 or payload.get("run_id") != run["run_id"]
                    or payload.get("score_name") != expected_score):
                raise ValueError(f"{path}: incompatible sample/run metadata.")
            if sample["record_index"] != index or path.stem != f"sample_{index:06d}":
                raise ValueError(f"{folder}: expected consecutive sample indices starting at zero.")
            if layers != list(range(28)):
                raise ValueError(f"{path}: this pilot requires collected layers 0-27.")
            source = sample["source_record"]
            if task == "U":
                steps, n_tokens = [-1], len(payload["tokens"])
                values = payload["layers"]["0"]
                width, dtype = values["input_features"].shape[1], values["input_features"].dtype
                if set(payload["layers"]) != set(map(str, layers)):
                    raise ValueError(f"{path}: missing collected U layers.")
                for values in payload["layers"].values():
                    if (values["input_features"].shape != (n_tokens, width)
                            or values["input_features"].dtype != dtype
                            or values["gradient_update"].shape != (n_tokens,)):
                        raise ValueError(f"{path}: incompatible feature/target shapes.")
                identities = [f"U/{kind}/{value}" for kind, value in (
                    ("image", sample.get("image_sha256")),
                    ("path", sample.get("image_path") or source.get("image_path")),
                    ("source", source.get("source_url"))) if value]
                if not str(source.get("expected_answer", "")).strip() or not identities:
                    raise ValueError(f"{path}: missing answer or image identity.")
            else:
                if payload.get("storage_format") != "gradient_update_generation_light_v1":
                    raise ValueError(f"{path}: expected lightweight generation data.")
                branch, steps = payload["branches"]["conditional"], payload["step_indices"]
                valid = branch["valid_token_mask"]
                scores = branch["gradient_update_tensor"]
                n_tokens, width = int(valid.sum()), int(run["backbone_config"]["hidden_size"])
                dtype = payload["initial_noise"].dtype
                if (valid.ndim != 1 or valid.dtype != torch.bool or
                        scores.shape != (len(layers), len(steps), len(valid))):
                    raise ValueError(f"{path}: incompatible generation score layout.")
                if (steps != run["effective_sampling"]["recorded_steps"] or
                        payload["denoising_timesteps"] != run["effective_sampling"]["timesteps"]):
                    raise ValueError(f"{path}: collection steps disagree with run.json.")
                if str(dtype).removeprefix("torch.") != run["dtype"].removeprefix("torch."):
                    raise ValueError(f"{path}: replay precision differs from collection.")
                prompt = " ".join(source["prompt"].split()).casefold()
                if not prompt:
                    raise ValueError(f"{path}: empty generation prompt.")
                identities = ["G/prompt/" + prompt]
            if n_tokens <= 0:
                raise ValueError(f"{path}: no valid tokens.")
            group.append(dict(uid=f"{task}/{category}/{path.stem}", task=task,
                              category=f"{task}/{category}", index=index, split="test",
                              cohort="original" if index < (6 if task == "U" else 12) else "new",
                              aggregate=str(path), run_file=str(run_file), steps=steps,
                              layers=layers, feature_dim=width,
                              dtype=str(dtype).removeprefix("torch."),
                              n_valid_tokens=n_tokens, source_record=source, identities=identities))
            del payload
        original_count, train_end, val_end = (6, 4, 5) if task == "U" else (12, 8, 10)
        for rank, position in enumerate(rng.permutation(original_count)):
            group[int(position)]["split"] = ("train" if rank < train_end else
                                             "val" if rank < val_end else "test")
        examples.extend(group)
    if len({(x["feature_dim"], x["dtype"]) for x in examples}) != 1:
        raise ValueError("U/G features must share width and precision.")
    seen = {}
    for example in examples:
        for identity in example["identities"]:
            if identity in seen and seen[identity]["split"] != example["split"]:
                raise ValueError(f"Data leakage: {seen[identity]['uid']} and {example['uid']}.")
            seen[identity] = example
    return examples


class FeatureSource:
    """Read U features; reconstruct one G step with the frozen UMM."""

    def __init__(self, examples, device):
        self.device, self.resources = torch.device(device), None
        self.g_run = next(x["run_file"] for x in examples if x["task"] == "G")

    def get(self, example, step=-1):
        if step not in example["steps"]:
            raise ValueError(f"Unrecorded step {step}: {example['uid']}")
        payload = load_saved(example["aggregate"])
        if example["task"] == "U":
            return {"layers": payload["layers"],
                    "valid": torch.ones(len(payload["tokens"]), dtype=torch.bool)}
        if self.resources is None:
            from omegaconf import OmegaConf
            from evaluation.calculate_token_importance_understanding import load_model_resources
            run = json.loads(Path(self.g_run).read_text(encoding="utf-8"))
            config = OmegaConf.create(run["effective_config"])
            config.device, config.dtype = str(self.device), run["dtype"].removeprefix("torch.")
            config.layers = run["selected_layers"]
            resources = load_model_resources(config, self.device, getattr(torch, config.dtype))
            resources.model.eval().requires_grad_(False)
            actual = getattr(resources.model.showo.model.config, "_attn_implementation", None)
            if run.get("attention_implementation") != actual:
                warnings.warn("Replay attention backend differs from the saved collection.")
            for key, backend in (("cuda_matmul_allow_tf32", torch.backends.cuda.matmul),
                                 ("cudnn_allow_tf32", torch.backends.cudnn)):
                if key in run:
                    backend.allow_tf32 = bool(run[key])
            self.resources = resources._replace(vae=None)
            del resources
            gc.collect()
            if self.device.type == "cuda":
                torch.cuda.empty_cache()
        from evaluation.calculate_gradient_update_generation_light import replay_step_features
        return {"layers": replay_step_features(self.resources, payload, step),
                "valid": payload["branches"]["conditional"]["valid_token_mask"]}

    def close(self):
        self.resources = None
        gc.collect()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()


def build_banks(examples, source, tokens_per_group=32, seed=123, layers=tuple(range(27))):
    """Keep a small random training sample in RAM; never read test features."""
    if not isinstance(tokens_per_group, int) or tokens_per_group < 1:
        raise ValueError("tokens_per_group must be a positive integer.")
    selected = [x for x in examples if x["split"] in ("train", "val")]
    banks, offsets = {}, {}
    for split in ("train", "val"):
        banks[split] = {}
        for task in ("U", "G"):
            items = [x for x in selected if x["split"] == split and x["task"] == task]
            rows = sum(min(tokens_per_group, x["n_valid_tokens"]) * len(layers) * len(x["steps"])
                       for x in items)
            if not rows:
                raise ValueError(f"No {task} examples in {split}.")
            banks[split][task] = {
                "hidden": torch.empty(rows, items[0]["feature_dim"], dtype=getattr(torch, items[0]["dtype"])),
                "layer": torch.empty(rows, dtype=torch.long),
                "score": torch.empty(rows, dtype=torch.float32),
                "group": torch.empty(rows, dtype=torch.long),
            }
            offsets[split, task] = 0
    rng, group_id = torch.Generator().manual_seed(seed), 0
    for number, example in enumerate(selected, 1):
        print(f"Features {number}/{len(selected)}: {example['uid']}", flush=True)
        bank = banks[example["split"]][example["task"]]
        for step_number, step in enumerate(example["steps"], 1):
            batch = source.get(example, step)
            valid_positions = batch["valid"].nonzero().flatten()
            if len(valid_positions) != example["n_valid_tokens"]:
                raise ValueError(f"Valid-token count changed: {example['uid']}")
            count = min(tokens_per_group, len(valid_positions))
            for layer in layers:
                values = batch["layers"][str(layer)]
                if (values["input_features"].shape != (len(batch["valid"]), example["feature_dim"])
                        or values["input_features"].dtype != bank["hidden"].dtype
                        or values["gradient_update"].shape != batch["valid"].shape):
                    raise ValueError(f"Replayed feature/target layout changed: {example['uid']}.")
                positions = valid_positions[torch.randperm(len(valid_positions), generator=rng)[:count]]
                hidden = values["input_features"][positions].detach().cpu()
                score = values["gradient_update"][positions].detach().float().cpu()
                if not torch.isfinite(hidden).all() or not torch.isfinite(score).all() or (score < 0).any():
                    raise ValueError(f"Invalid features or targets: {example['uid']}, layer {layer}.")
                start = offsets[example["split"], example["task"]]
                stop = start + count
                bank["hidden"][start:stop], bank["score"][start:stop] = hidden, score
                bank["layer"][start:stop], bank["group"][start:stop] = layer, group_id
                offsets[example["split"], example["task"]], group_id = stop, group_id + 1
            del batch, values, hidden, score
            if len(example["steps"]) > 1 and (step_number % 10 == 0 or step_number == len(example["steps"])):
                print(f"  {step_number}/{len(example['steps'])} steps", flush=True)
    return banks["train"], banks["val"]

In [ ]:
examples = discover_data(U_ROOT, G_ROOT, U_CATEGORIES, SPLIT_SEED)
FEATURE_DIM = examples[0]["feature_dim"]
display(pd.DataFrame(examples).groupby(["task", "split", "cohort"]).size().rename("examples").to_frame())

# Identify the saved data before reusing weights.
files = sorted({x[key] for x in examples for key in ["aggregate", "run_file"]})
stamp = [(path, Path(path).stat().st_size, Path(path).stat().st_mtime_ns) for path in files]
settings = dict(version="minimal_v1", files=stamp, seeds=SEEDS, split_seed=SPLIT_SEED,
                token_seed=TOKEN_SEED, tokens_per_group=TOKENS_PER_GROUP, layers=list(LAYERS),
                depth_start=DEPTH_START, top_fraction=TOP_FRACTION, train=TRAIN,
                feature_dim=FEATURE_DIM, target="log(clamp_min(raw,1e-12))")
signature = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()

train_rows = sum(min(TOKENS_PER_GROUP, x["n_valid_tokens"]) * len(LAYERS) * len(x["steps"])
                 for x in examples if x["split"] != "test")
element_bytes = torch.empty((), dtype=getattr(torch, examples[0]["dtype"])).element_size()
bank_bytes = train_rows * (FEATURE_DIM * element_bytes + 20)
print(f"Compact training + validation tensors: {bank_bytes / 2**30:.2f} GiB")
if bank_bytes > 2 * 2**30:
    raise MemoryError("Reduce TOKENS_PER_GROUP; the compact tensors exceed the 2 GiB budget.")

These are small pilot sets, not balanced benchmarks. For example, the first ten traffic-light records all answer “red.” Report original and new test cohorts separately.

### 1.4 Define the shared scorer and corrections

`f` learns from balanced U/G batches. Freeze `f`; train `u` on U and `g` on G to predict **`log_target − f`**. Both corrections train on all layers 0–26. The UMM stays frozen.

The input is the normalized hidden state, log RMS, and layer index. There is no generation-progress correction in this pilot.

In [ ]:
import torch
from torch import nn


class TokenMLP(nn.Module):
    """Score each token using its hidden state and layer index."""

    def __init__(self, feature_dim, hidden_width=128, num_layers=28):
        super().__init__()
        self.feature_dim, self.num_layers = feature_dim, num_layers
        self.net = nn.Sequential(
            nn.Linear(feature_dim + 1 + num_layers, hidden_width),
            nn.GELU(), nn.Linear(hidden_width, 1),
        )

    def forward(self, hidden, layer):
        hidden = hidden.float()
        rms = hidden.square().mean(1, keepdim=True).sqrt().clamp_min(1e-6)
        layer_code = nn.functional.one_hot(layer.long(), self.num_layers).float()
        return self.net(torch.cat([hidden / rms, rms.log(), layer_code], 1)).squeeze(1)


def _prepare_bank(bank):
    """Give every example/layer/step group equal sampling weight."""
    _, inverse, counts = torch.unique(bank["group"], return_inverse=True, return_counts=True)
    if not len(counts) or (bank["layer"] < 0).any() or (bank["layer"] > 26).any():
        raise ValueError("Training needs nonempty groups from layers 0–26.")
    return dict(bank, inverse=inverse, counts=counts,
                order=torch.argsort(inverse, stable=True), starts=counts.cumsum(0) - counts)


def _sample_rows(bank, count, generator):
    groups = torch.randint(len(bank["counts"]), (count,), generator=generator)
    offsets = (torch.rand(count, generator=generator) * bank["counts"][groups]).long()
    indices = bank["order"][bank["starts"][groups] + offsets]
    return tuple(bank[key][indices] for key in ("hidden", "layer", "score"))


def _training_target(score, hidden, layer, core):
    target = score.float().clamp_min(1e-12).log()
    if core is not None:
        with torch.no_grad():
            target = target - core(hidden, layer)
    return target


@torch.no_grad()
def _validation_loss(model, core, banks, device):
    model.eval()
    task_losses = []
    for bank in banks.values():
        total = 0.0
        for start in range(0, len(bank["score"]), 4096):
            section = slice(start, start + 4096)
            hidden, layer, score = (bank[key][section].to(device) for key in ("hidden", "layer", "score"))
            target = _training_target(score, hidden, layer, core)
            error = (model(hidden, layer) - target).square().double().cpu()
            weights = bank["counts"][bank["inverse"][section]].double()
            total += (error / weights).sum().item() / len(bank["counts"])
        task_losses.append(total)
    return sum(task_losses) / len(task_losses)


def train_scorer(name, seed, train, val, feature_dim, core=None, device="cpu",
                 hidden_width=128, batch_size=256, max_epochs=15, updates=150,
                 patience=3, lr=1e-3):
    """Train f, u or g; retain the epoch with the lowest validation MSE."""
    if name not in ("f", "u", "g") or ((name == "f") != (core is None)):
        raise ValueError("Train f alone; supply its frozen model when training u or g.")
    if batch_size < 2 or batch_size % 2 or min(max_epochs, updates, patience) < 1:
        raise ValueError("Use an even batch size and positive training settings.")
    tasks = ("U", "G") if name == "f" else (name.upper(),)
    training = {task: _prepare_bank(train[task]) for task in tasks}
    validation = {task: _prepare_bank(val[task]) for task in tasks}
    if core is not None:
        core.to(device).eval()
        core.requires_grad_(False)
        for parameter in core.parameters():
            parameter.grad = None
    torch.manual_seed(seed)
    model = TokenMLP(feature_dim, hidden_width).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(seed + {"f": 0, "u": 1000, "g": 2000}[name])
    best_loss, stale, best_state, history = float("inf"), 0, None, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for _ in range(updates):
            batches = [_sample_rows(training[task], batch_size // len(tasks), generator) for task in tasks]
            hidden, layer, score = (torch.cat([batch[i] for batch in batches]).to(device) for i in range(3))
            target = _training_target(score, hidden, layer, core)
            loss = nn.functional.mse_loss(model(hidden, layer), target)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss; inspect features and targets.")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        val_loss = _validation_loss(model, core, validation, device)
        if not torch.isfinite(torch.tensor(val_loss)):
            raise ValueError("Nonfinite validation loss; inspect features and targets.")
        history.append(dict(component=name, seed=seed, epoch=epoch,
                            train_loss=epoch_loss / updates, val_loss=val_loss))
        print(f"{name}, seed {seed}, epoch {epoch}: validation MSE = {val_loss:.4f}")
        if val_loss < best_loss:
            best_loss, stale = val_loss, 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            stale += 1
        if stale >= patience:
            break
    model.load_state_dict(best_state)
    return model.eval(), history

### 1.5 Train once, then save or reload

An existing `routers.pt` is reused only when data and settings match. Otherwise choose a new `OUTPUT_DIR`. No full feature archive is written: sampled features live only in RAM and are released after training.

In [ ]:
models, history = {}, []
if WEIGHTS.exists():
    saved = torch.load(WEIGHTS, map_location="cpu", weights_only=True)
    if saved["signature"] != signature:
        raise ValueError("Saved weights use different data/settings. Choose a new OUTPUT_DIR.")
    for seed in SEEDS:
        models[seed] = {}
        for name in ["f", "u", "g"]:
            model = TokenMLP(FEATURE_DIM, TRAIN["hidden_width"])
            model.load_state_dict(saved["models"][seed][name])
            models[seed][name] = model.to(DEVICE).eval()
    history = saved["history"]
    print("Loaded matching scorers:", WEIGHTS)
else:
    source = FeatureSource(examples, DEVICE)
    try:
        train_banks, val_banks = build_banks(examples, source, TOKENS_PER_GROUP, TOKEN_SEED, LAYERS)
    finally:
        source.close()
    for seed in SEEDS:
        core, rows = train_scorer("f", seed, train_banks, val_banks, FEATURE_DIM, device=DEVICE, **TRAIN)
        models[seed] = {"f": core}
        history.extend(rows)
        for name in ["u", "g"]:
            model, rows = train_scorer(name, seed, train_banks, val_banks, FEATURE_DIM,
                                       core=core, device=DEVICE, **TRAIN)
            models[seed][name] = model
            history.extend(rows)
    saved = dict(signature=signature, settings=settings, history=history, models={
        seed: {name: {key: value.detach().cpu() for key, value in model.state_dict().items()}
               for name, model in family.items()} for seed, family in models.items()
    })
    temporary = WEIGHTS.with_suffix(".tmp")
    torch.save(saved, temporary)
    temporary.replace(WEIGHTS)
    del train_banks, val_banks
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
display(pd.DataFrame(history).groupby(["component", "seed"])["val_loss"].min().rename("best_validation_MSE"))

## 2. Compare token rankings
### 2.1 Define the three configurations

| Configuration | Layers 0–13 | Layers 14–26 |
|---|---|---|
| Shared everywhere | `f` | `f` |
| Specialized everywhere | `f + u/g` | `f + u/g` |
| Depth-dependent | `f` | `f + u/g` |

Use `u` for U and `g` for G, with the **same trained weights** across configurations. The depth curve equals the shared curve below layer 14 and the specialized curve from layer 14 by definition.

Evaluate all valid tokens using `K = ceil(20% × valid token count)`. Overlap is the intersection of predicted and raw-target top-K sets divided by K. Stable token order breaks ties; ambiguous cutoffs are reported. Layer 27 is excluded for both tasks.

In [ ]:
import math
import numpy as np
import pandas as pd


MODES = ("Shared everywhere", "Specialized everywhere", "Depth-dependent")


@torch.inference_mode()
def evaluate(examples, source, models, split="val", depth_start=14,
             layers=tuple(range(27)), device="cpu", top_fraction=0.2, progress=print):
    """Compare rankings on every valid token; this does not run sparse inference."""
    selected = [example for example in examples if example["split"] == split]
    if not selected or not models or not 0 < top_fraction <= 1:
        raise ValueError("Supply models, a nonempty split, and 0 < top_fraction <= 1.")
    if not layers or min(layers) < 0 or max(layers) > 26 or not 0 <= depth_start <= 27:
        raise ValueError("Evaluate layers 0–26 with a depth boundary between 0 and 27.")
    for family in models.values():
        for model in family.values():
            model.to(device).eval()
    rows = []
    for number, example in enumerate(selected, 1):
        if progress:
            progress(f"{split}: {number}/{len(selected)} — {example['uid']}")
        for step_number, step in enumerate(example["steps"], 1):
            payload = source.get(example, step)
            valid = torch.as_tensor(payload["valid"], dtype=torch.bool).cpu().reshape(-1)
            n_tokens = int(valid.sum())
            if not n_tokens:
                raise ValueError(f"No valid tokens: {example['uid']}, step {step}.")
            k = math.ceil(top_fraction * n_tokens)
            for layer in layers:
                block = payload["layers"][str(layer)]
                truth = block["gradient_update"].detach().cpu().reshape(-1)[valid].float().numpy()
                hidden = block["input_features"].detach().cpu()[valid].to(device)
                if not np.isfinite(truth).all() or (truth < 0).any() or not torch.isfinite(hidden).all():
                    raise ValueError(f"Invalid features or raw targets: {example['uid']}, layer {layer}.")
                target_top = np.argsort(-truth, kind="stable")[:k]
                cutoff = truth[target_top[-1]]
                ambiguous = int((truth == cutoff).sum()) > k - int((truth > cutoff).sum())
                layer_ids = torch.full((n_tokens,), layer, dtype=torch.long, device=device)
                for seed, family in models.items():
                    shared = family["f"](hidden, layer_ids)
                    correction = family["u" if example["task"] == "U" else "g"](hidden, layer_ids)
                    specialized = shared + correction
                    predictions = (shared, specialized, specialized if layer >= depth_start else shared)
                    for mode, scores in zip(MODES, predictions):
                        scores = scores.float().cpu().numpy()
                        if scores.shape != (n_tokens,) or not np.isfinite(scores).all():
                            raise ValueError("Router returned invalid token scores.")
                        predicted_top = np.argsort(-scores, kind="stable")[:k]
                        rows.append({
                            **{key: example[key] for key in ("uid", "task", "category", "split", "cohort")},
                            "seed": seed, "layer": layer, "step": step, "mode": mode,
                            "overlap": len(np.intersect1d(predicted_top, target_top)) / k,
                            "chance": k / n_tokens, "n_tokens": n_tokens, "k": k,
                            "target_cutoff_ambiguous": ambiguous,
                            "target_zero_fraction": float((truth == 0).mean()),
                        })
                del hidden
            del payload
            if progress and len(example["steps"]) > 1 and (step_number % 10 == 0 or step_number == len(example["steps"])):
                progress(f"  {step_number}/{len(example['steps'])} steps")
    return pd.DataFrame(rows)


def summarize(details):
    """Average steps, layers and seeds; then give each example equal weight."""
    keys, values = ["task", "cohort", "mode"], ["overlap", "chance"]
    by_layer = details.groupby(keys + ["uid", "seed", "layer"], as_index=False)[values].mean()
    by_seed = by_layer.groupby(keys + ["uid", "seed"], as_index=False)[values].mean()
    by_example = by_seed.groupby(keys + ["uid"], as_index=False)[values].mean()
    result = by_example.groupby(keys, as_index=False).agg(
        mean_overlap=("overlap", "mean"), mean_chance=("chance", "mean"), n_examples=("uid", "nunique"),
    )
    seeds = details.groupby(keys, as_index=False).agg(n_seeds=("seed", "nunique"))
    return result.merge(seeds, on=keys, validate="one_to_one")

### 2.2 Evaluate validation, then test

Keep settings fixed before inspecting test results. Training seeds are repeated runs, not additional test examples. Each example has equal weight after averaging steps, layers, and seeds.

In [ ]:
source = FeatureSource(examples, DEVICE)
try:
    for split in ["val", "test"]:
        details = evaluate(examples, source, models, split=split, depth_start=DEPTH_START,
                           layers=LAYERS, device=DEVICE, top_fraction=TOP_FRACTION)
        details.to_csv(OUTPUT_DIR / f"{split}_overlap.csv", index=False)
        summary = summarize(details)
        summary.to_csv(OUTPUT_DIR / f"{split}_summary.csv", index=False)
        print(split)
        display(summary.round(4))
finally:
    source.close()

### 2.3 Plot the new held-out examples

After a runtime reset, run **1.1, 1.2, and this cell** to redraw from the saved CSV without replay or training. Curves use the new test cohort; raw CSVs retain both cohorts. Larger deep-layer gains from specialization would motivate the inference experiment, but do not establish speedup.

In [ ]:
details = pd.read_csv(OUTPUT_DIR / "test_overlap.csv")
data = details[details["cohort"] == "new"].copy()
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "legend.fontsize": 9, "pdf.fonttype": 42})
styles = {"Shared everywhere": ("#56B4E9", "o", "-"),
          "Specialized everywhere": ("#00A69D", "s", "-"),
          "Depth-dependent": ("#CC79A7", "^", "--")}

def curves(data, axis):
    values = ["overlap", "chance"]
    grouped = data.groupby(["task", "mode", "uid", "seed", axis], as_index=False)[values].mean()
    grouped = grouped.groupby(["task", "mode", "uid", axis], as_index=False)[values].mean()
    return grouped.groupby(["task", "mode", axis], as_index=False)[values].mean()

def draw(ax, table, axis, title):
    for mode, (color, marker, line) in styles.items():
        part = table[table["mode"] == mode].sort_values(axis)
        ax.plot(part[axis], part["overlap"], color=color, marker=marker, linestyle=line,
                markersize=3.5, linewidth=1.8, label=mode)
    chance = table[table["mode"] == "Shared everywhere"]["chance"].mean()
    ax.axhline(chance, color="black", linestyle="--", linewidth=1.3, label=f"Random ≈ {chance:.1%}")
    ax.set(title=title, xlabel="Layer" if axis == "layer" else "Generation step", ylim=(0, 1))
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.2)

by_layer = curves(data, "layer")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True, constrained_layout=True)
for ax, task, title in zip(axes, ["U", "G"], ["Understanding", "Generation (steps averaged)"]):
    draw(ax, by_layer[by_layer["task"] == task], "layer", title + " — new test")
    ax.axvline(DEPTH_START - 0.5, color="black", alpha=0.25, linewidth=0.8)
axes[0].set_ylabel("Top-20% overlap with Gradient × Update")
axes[1].legend(frameon=False)
fig.savefig(OUTPUT_DIR / "overlap_by_layer.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
draw(ax, curves(data[data["task"] == "G"], "step"), "step", "Generation — new test (layers 0–26 averaged)")
ax.set_ylabel("Top-20% overlap with Gradient × Update")
ax.legend(frameon=False)
fig.savefig(OUTPUT_DIR / "overlap_by_step.pdf", bbox_inches="tight")
plt.show()

ties = data.drop_duplicates(["uid", "layer", "step"])
display(ties.groupby("task")[["target_cutoff_ambiguous", "target_zero_fraction"]].mean().round(3))
print("Saved results:", OUTPUT_DIR)

## 3. Apply actual FFN routing
### 3.1 Sparse FFN adapter

This adapter performs real gather/scatter computation:

1. Score the eligible tokens from each decoder block's **input**, matching training.
2. Run that layer's original FFN only for the highest-scoring 80%.
3. Scatter those FFN outputs back; skipped rows receive zero FFN output and keep the residual path.

Attention still processes every token. Text, special tokens, padding, the CFG unconditional branch, and layer 27 remain dense. The 20% skip therefore applies to **eligible image tokens**, not to every sequence row.

In [ ]:
import math
import torch


class FFNRouter:
    """Route FFN rows using block-input scores; keep attention unchanged."""

    def __init__(self, core, scorers, task, mode, eligible_mask,
                 keep_fraction=0.8, depth_start=14):
        if task not in ("U", "G") or mode not in ("dense", "shared", "specialized", "depth"):
            raise ValueError("Use task U/G and mode dense/shared/specialized/depth.")
        if not 0 < keep_fraction <= 1 or not 0 <= depth_start < 27:
            raise ValueError("Require 0 < keep_fraction <= 1 and depth_start in 0..26.")
        if eligible_mask.ndim != 2 or eligible_mask.dtype != torch.bool:
            raise ValueError("eligible_mask must be bool [batch, tokens].")
        if len(core.layers) != 28:
            raise ValueError("This pilot expects the 28-layer Show-o2 backbone.")
        self.core, self.scorers, self.task, self.mode = core, scorers, task, mode
        self.keep_fraction, self.depth_start = keep_fraction, depth_start
        self.shape = tuple(eligible_mask.shape)
        device = next(core.parameters()).device
        mask = eligible_mask.detach().cpu()
        self.eligible = [row.nonzero(as_tuple=True)[0].to(device) for row in mask]
        self.protected = (~mask).flatten().nonzero(as_tuple=True)[0].to(device)
        self.keep = [math.ceil(keep_fraction * positions.numel()) for positions in self.eligible]
        self.n_eligible = sum(positions.numel() for positions in self.eligible)
        self.n_kept = sum(self.keep)
        self.stats = dict(calls=0, rows_total=0, rows_computed=0,
                          eligible_rows=0, selected_eligible=0)
        self.by_layer, self._hooks, self._originals, self._selected = {}, [], [], {}

    def __enter__(self):
        if self.core.training or any(layer.training for layer in self.core.layers):
            raise ValueError("Use model.eval() before inference routing.")
        if hasattr(self.core, "_depth_router_active"):
            raise RuntimeError("A routing context is already active on this backbone.")
        if getattr(self.core, "is_hybrid_pruned", False) or any(
            hasattr(layer.mlp, "gate_proj_pruned") for layer in self.core.layers
        ):
            raise ValueError("Use the original dense FFN weights for this comparison.")
        self.core._depth_router_active = self
        try:
            if self.mode == "dense":
                for index, layer in enumerate(self.core.layers[:-1]):
                    self._hooks.append(layer.register_forward_pre_hook(
                        self._make_dense_counter(index), with_kwargs=True))
                return self
            if self.keep_fraction == 1 or self.n_eligible == 0:
                return self
            required = ["f"] + ([self.task.lower()] if self.mode != "shared" else [])
            if any(self.scorers[name].training for name in required):
                raise ValueError("Use scorer.eval() before inference routing.")
            for index, layer in enumerate(self.core.layers[:-1]):
                self._hooks.append(layer.register_forward_pre_hook(
                    self._make_selector(index), with_kwargs=True))
                original = layer.mlp.forward
                self._originals.append((layer.mlp, original))
                layer.mlp.forward = self._make_sparse_ffn(index, original)
            return self
        except Exception:
            self.__exit__(None, None, None)
            raise

    def _make_dense_counter(self, index):
        def count(module, args, kwargs):
            hidden = args[0] if args else kwargs["hidden_states"]
            if tuple(hidden.shape[:2]) != self.shape:
                raise ValueError("Routing mask must match this forward's batch and token count.")
            rows = hidden.shape[0] * hidden.shape[1]
            counts = dict(calls=1, rows_total=rows, rows_computed=rows,
                          eligible_rows=self.n_eligible, selected_eligible=self.n_eligible)
            layer_counts = self.by_layer.setdefault(index, dict.fromkeys(counts, 0))
            for key, value in counts.items():
                self.stats[key] += value
                layer_counts[key] += value
        return count

    def _make_selector(self, index):
        @torch.no_grad()
        def select(module, args, kwargs):
            hidden = args[0] if args else kwargs["hidden_states"]
            if tuple(hidden.shape[:2]) != self.shape:
                raise ValueError("Routing mask must match this forward's batch and token count.")
            selected = [self.protected]
            for batch, (positions, count) in enumerate(zip(self.eligible, self.keep)):
                if count == 0:
                    continue
                features = hidden[batch].index_select(0, positions)
                layers = torch.full((len(positions),), index, device=hidden.device, dtype=torch.long)
                scores = self.scorers["f"](features, layers)
                if self.mode == "specialized" or (self.mode == "depth" and index >= self.depth_start):
                    scores = scores + self.scorers[self.task.lower()](features, layers)
                order = torch.argsort(scores, descending=True, stable=True)[:count]
                selected.append(positions.index_select(0, order) + batch * self.shape[1])
            self._selected[index] = torch.cat(selected)
        return select

    def _make_sparse_ffn(self, index, original):
        def sparse(hidden):
            selected = self._selected.pop(index)
            if tuple(hidden.shape[:2]) != self.shape:
                raise ValueError("FFN input does not match its block-input routing mask.")
            flat = hidden.reshape(-1, hidden.shape[-1])
            output = torch.zeros_like(flat)
            output.index_copy_(0, selected, original(flat.index_select(0, selected)))
            counts = dict(calls=1, rows_total=flat.shape[0], rows_computed=selected.numel(),
                          eligible_rows=self.n_eligible, selected_eligible=self.n_kept)
            layer_counts = self.by_layer.setdefault(index, dict.fromkeys(counts, 0))
            for key, value in counts.items():
                self.stats[key] += value
                layer_counts[key] += value
            return output.reshape_as(hidden)
        return sparse

    def __exit__(self, exc_type, exc_value, traceback):
        for module, original in reversed(self._originals):
            module.forward = original
        for hook in self._hooks:
            hook.remove()
        self._originals.clear()
        self._hooks.clear()
        self._selected.clear()
        if getattr(self.core, "_depth_router_active", None) is self:
            del self.core._depth_router_active
        return False

### 3.2 Actual inference functions

Understanding uses one routed prompt prefill followed by dense cached answer decoding. Generation uses the repository's real ODE sampler from the saved initial noise. The light-data replay path is not used as inference.

The functions verify that rebuilt U/G inputs match the saved collection layout. They never append the known answer during inference.

In [ ]:
@torch.no_grad()
def load_routing_resources(g_example, device):
    """Load one complete frozen UMM, including the VAE used to decode G outputs."""
    from omegaconf import OmegaConf
    from evaluation.calculate_token_importance_understanding import load_model_resources

    run = json.loads(Path(g_example["run_file"]).read_text(encoding="utf-8"))
    config = OmegaConf.create(run["effective_config"])
    config.device = str(device)
    config.dtype = run["dtype"].removeprefix("torch.")
    config.layers = "all"
    resources = load_model_resources(config, torch.device(device), getattr(torch, config.dtype))
    if len(resources.model.showo.model.layers) != 28:
        raise ValueError("This pilot expects the 28-layer Show-o2 backbone.")
    return resources, config, run


def image_mask(batch, length, batch_index, positions):
    """Only image tokens are eligible; all other rows stay dense."""
    mask = torch.zeros(batch, length, dtype=torch.bool)
    mask[batch_index, torch.as_tensor(positions, dtype=torch.long).cpu()] = True
    return mask


@torch.no_grad()
def routed_understanding(example, resources, scorers, mode, keep_fraction=0.8,
                         depth_start=14, max_new_tokens=16):
    """Route prompt prefill once, then decode answer tokens densely with KV cache."""
    from transformers.cache_utils import DynamicCache
    from models import omni_attn_mask_naive
    from models.misc import interpolate_pos_encoding
    from evaluation.calculate_token_importance_understanding import assemble_prefill, seed_example

    payload = load_saved(example["aggregate"])
    sample = payload["sample"]
    seed_example(int(sample["seed"]), resources.device)
    embeds, attention, visual, layout = assemble_prefill(
        resources.model, resources.tokenizer, resources.token_ids,
        payload["image_latents"].to(resources.device), sample["source_record"]["prompt"],
        int(sample["sequence_length"]), omni_attn_mask_naive, interpolate_pos_encoding,
        truncate_prompt=bool(sample.get("prompt_truncated", False)),
    )
    rebuilt = [(token["token_type"], token["token_id"]) for token in layout["tokens"]]
    saved = [(token["token_type"], token["token_id"]) for token in payload["tokens"]]
    if rebuilt != saved or not torch.equal(visual.cpu(), payload["visual_token_positions"].cpu()):
        raise ValueError("Rebuilt understanding input differs from the saved collection layout.")

    eligible = image_mask(1, embeds.shape[1], 0, visual)
    cache = DynamicCache()
    router = FFNRouter(resources.model.showo.model, scorers, "U", mode, eligible,
                       keep_fraction, depth_start)
    with router:
        output = resources.model.showo(
            inputs_embeds=embeds, attention_mask=attention,
            past_key_values=cache, use_cache=True, return_dict=True,
            num_logits_to_keep=1,
        )

    stop_ids = {value for value in [resources.tokenizer.eos_token_id,
                resources.token_ids.get("eos_id")] if value is not None}
    im_end = resources.tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end is not None and im_end >= 0:
        stop_ids.add(im_end)
    generated = []
    decode_mask = torch.cat([
        attention[:, :, -1:, :],
        torch.zeros_like(attention[:, :, -1:, :1]),
    ], dim=-1)
    for _ in range(max_new_tokens):
        next_token = output.logits[:, -1].argmax(-1, keepdim=True)
        token_id = int(next_token.item())
        if token_id in stop_ids:
            break
        generated.append(token_id)
        output = resources.model.showo(
            input_ids=next_token, attention_mask=decode_mask,
            past_key_values=output.past_key_values, use_cache=True,
            return_dict=True, num_logits_to_keep=1,
        )
        decode_mask = torch.cat([
            decode_mask, torch.zeros_like(decode_mask[..., :1])
        ], dim=-1)
    return {
        "uid": example["uid"], "task": "U", "mode": mode,
        "answer": resources.tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "token_ids": generated, "eligible_image_tokens": int(eligible.sum()),
        "kept_image_tokens_per_layer": int(eligible.sum().item() if mode == "dense" else
                                                   np.ceil(keep_fraction * eligible.sum().item())),
        "router_stats": router.stats, "router_by_layer": router.by_layer,
    }


@torch.no_grad()
def routed_generation(example, resources, config, run, scorers, mode, output_path,
                      keep_fraction=0.8, depth_start=14):
    """Run the genuine guided image sampler with conditional image-token routing."""
    from PIL import Image
    from utils import denorm
    from evaluation.calculate_token_importance_generation import (
        prepare_generation_inputs, create_generation_sampler,
    )
    from evaluation.calculate_token_importance_understanding import seed_example

    payload = load_saved(example["aggregate"])
    sample = payload["sample"]
    seed_example(int(sample["seed"]), resources.device)
    guidance = float(sample["reference_guidance_scale"])
    inputs = prepare_generation_inputs(
        sample["source_record"], resources, config, {"guidance_scale": guidance}
    )
    replay = payload["replay_inputs"]
    if (not torch.equal(inputs["text_tokens"][0].cpu(), replay["text_tokens"][0].cpu()) or
            not torch.equal(inputs["modality_positions"][0].cpu(),
                            replay["modality_positions"][0].cpu())):
        raise ValueError("Rebuilt generation input differs from the saved conditional layout.")

    conditional = inputs["branches"]["conditional"]
    eligible = image_mask(
        inputs["text_tokens"].shape[0], inputs["sequence_length"],
        int(conditional["batch_index"]), conditional["visual_token_positions"],
    )
    noise = payload["initial_noise"].to(resources.device)
    sampler_noise = torch.cat([noise, noise]) if guidance > 0 else noise
    sampler = create_generation_sampler(run["effective_sampling"])
    router = FFNRouter(resources.model.showo.model, scorers, "G", mode, eligible,
                       keep_fraction, depth_start)
    with router:
        trajectory = sampler(
            sampler_noise, resources.model.t2i_generate,
            text_tokens=inputs["text_tokens"], attention_mask=inputs["attention_mask"],
            modality_positions=inputs["modality_positions"],
            max_seq_len=inputs["sequence_length"], guidance_scale=guidance,
        )
    latent = trajectory[-1, :1].to(resources.dtype)
    pixels = resources.vae.batch_decode(latent.unsqueeze(2)).squeeze(2)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(denorm(pixels)[0]).save(output_path)
    return {
        "uid": example["uid"], "task": "G", "mode": mode,
        "image": str(output_path), "guidance_scale": guidance,
        "eligible_image_tokens": int(eligible.sum()),
        "kept_image_tokens_per_layer": int(eligible.sum().item() if mode == "dense" else
                                                   np.ceil(keep_fraction * eligible.sum().item())),
        "unconditional_branch_dense": bool(guidance > 0),
        "router_stats": router.stats, "router_by_layer": router.by_layer,
    }

### 3.3 Run a functional routed-inference pilot

The default runs one new held-out U example and one new held-out G example through dense, shared, specialized, and depth-dependent modes using router seed 42. Increase `STEP3_EXAMPLES_PER_TASK` after the first successful run.

This cell saves generated answers, images, and exact FFN row counts. It does not claim a speedup; synchronized timing and output-quality evaluation belong to step 4.

In [ ]:
RUN_STEP3 = True
STEP3_EXAMPLES_PER_TASK = 1
STEP3_ROUTER_SEED = 42
STEP3_MODES = ["dense", "shared", "specialized", "depth"]
STEP3_MAX_NEW_TOKENS = 16
STEP3_DIR = OUTPUT_DIR / "routed_inference"

In [ ]:
if RUN_STEP3:
    if STEP3_ROUTER_SEED not in models:
        raise ValueError("STEP3_ROUTER_SEED must identify one trained f/u/g family.")
    routed_examples = {
        task: [x for x in examples if x["task"] == task and x["split"] == "test"
               and x["cohort"] == "new"][:STEP3_EXAMPLES_PER_TASK]
        for task in ["U", "G"]
    }
    if any(len(items) != STEP3_EXAMPLES_PER_TASK for items in routed_examples.values()):
        raise ValueError("Not enough new test examples for the requested step-3 pilot.")

    g_example = routed_examples["G"][0]
    routing_resources, routing_config, g_run = load_routing_resources(g_example, DEVICE)
    scorers = models[STEP3_ROUTER_SEED]
    results = []
    try:
        for example in routed_examples["U"]:
            for mode in STEP3_MODES:
                print("U", example["uid"], mode, flush=True)
                result = routed_understanding(
                    example, routing_resources, scorers, mode,
                    KEEP_FRACTION, DEPTH_START, STEP3_MAX_NEW_TOKENS,
                )
                results.append(result)
                print("  answer:", result["answer"])
        for example in routed_examples["G"]:
            for mode in STEP3_MODES:
                name = f"G_{example['index']:06d}_{mode}.png"
                print("G", example["uid"], mode, flush=True)
                result = routed_generation(
                    example, routing_resources, routing_config, g_run,
                    scorers, mode, STEP3_DIR / name,
                    KEEP_FRACTION, DEPTH_START,
                )
                results.append(result)
                print("  image:", result["image"])
    finally:
        del routing_resources
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    (STEP3_DIR / "routing_results.json").write_text(
        json.dumps(results, indent=2), encoding="utf-8"
    )
    routing_table = pd.DataFrame([{
        "uid": row["uid"], "task": row["task"], "mode": row["mode"],
        "eligible_image_tokens": row["eligible_image_tokens"],
        "kept_per_routed_layer": row["kept_image_tokens_per_layer"],
        "routable_layer_rows": row["router_stats"]["rows_total"],
        "routable_layer_ffn_rows_computed": row["router_stats"]["rows_computed"],
    } for row in results])
    routing_table.to_csv(STEP3_DIR / "routing_compute.csv", index=False)
    display(routing_table)
    print("Step-3 outputs:", STEP3_DIR)

## 4. Measure quality and inference time — next

**Not implemented yet.** Step 3 now performs actual routed UMM inference and records the FFN rows computed. Step 4 must compare dense and routed modes on identical examples/seeds using:

- U answer accuracy;
- G prompt adherence and image quality;
- synchronized end-to-end inference time, including router and selection overhead.

A lower FFN row count alone is not proof of wall-clock speedup.